In [ ]:
!wget https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz




--2024-12-11 13:44:47--  https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
Resolving downloads.apache.org (downloads.apache.org)... 135.181.214.104, 88.99.208.237, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|135.181.214.104|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2024-12-11 13:44:47 ERROR 404: Not Found.



In [ ]:
!ls


sample_data


In [ ]:
!wget https://dlcdn.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz

--2024-12-11 13:50:17--  https://dlcdn.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
Resolving dlcdn.apache.org (dlcdn.apache.org)... 151.101.2.132, 2a04:4e42::644
Connecting to dlcdn.apache.org (dlcdn.apache.org)|151.101.2.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400864419 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.3-bin-hadoop3.tgz’

spark-3.5.3-bin-had 100%[===================>] 382.29M  32.1MB/s    in 6.8s    

2024-12-11 13:50:24 (55.9 MB/s) - ‘spark-3.5.3-bin-hadoop3.tgz’ saved [400864419/400864419]



In [ ]:
!tar xf spark-3.5.3-bin-hadoop3.tgz


In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"




In [47]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, desc

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Crime Data Analysis") \
    .getOrCreate()

# Step 2: Load the Crime Data
file_path = "/content/Crime_Data_from_2020_to_Present.csv"
crime_data = spark.read.csv(file_path, header=True, inferSchema=True)

# Step 3: Data Exploration
print("Schema of the dataset:")
crime_data.printSchema()

print("Sample data:")
crime_data.show(5)

# Step 4: Data Cleaning (if needed)
# Remove rows with null values in important columns
cleaned_data = crime_data.dropna(subset=["DATE OCC", "Crm Cd Desc", "LOCATION"])

# Step 5: Add Year and Month Columns
processed_data = cleaned_data.withColumn("Year", year(col("DATE OCC"))) \
    .withColumn("Month", month(col("DATE OCC")))

# Step 6: Aggregation - Crimes by Year
crimes_by_year = processed_data.groupBy("Year").count().orderBy("Year")
print("Crimes by Year:")
crimes_by_year.show()



Schema of the dataset:
root
 |-- DR_NO: integer (nullable = true)
 |-- Date Rptd: string (nullable = true)
 |-- DATE OCC: string (nullable = true)
 |-- TIME OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA NAME: string (nullable = true)
 |-- Rpt Dist No: integer (nullable = true)
 |-- Part 1-2: integer (nullable = true)
 |-- Crm Cd: integer (nullable = true)
 |-- Crm Cd Desc: string (nullable = true)
 |-- Mocodes: string (nullable = true)
 |-- Vict Age: integer (nullable = true)
 |-- Vict Sex: string (nullable = true)
 |-- Vict Descent: string (nullable = true)
 |-- Premis Cd: integer (nullable = true)
 |-- Premis Desc: string (nullable = true)
 |-- Weapon Used Cd: integer (nullable = true)
 |-- Weapon Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status Desc: string (nullable = true)
 |-- Crm Cd 1: integer (nullable = true)
 |-- Crm Cd 2: integer (nullable = true)
 |-- Crm Cd 3: integer (nullable = true)
 |-- Crm Cd 4: integer (

In [48]:
# Step 7: Aggregation - Top 10 Crime Categories
top_categories = processed_data.groupBy("Crm Cd Desc").count().orderBy(desc("count")).limit(10)
print("Top 10 Crime Categories:")
top_categories.show()



Top 10 Crime Categories:
+--------------------+------+
|         Crm Cd Desc| count|
+--------------------+------+
|    VEHICLE - STOLEN|111867|
|BATTERY - SIMPLE ...| 74244|
|BURGLARY FROM VEH...| 62010|
|   THEFT OF IDENTITY| 61141|
|VANDALISM - FELON...| 60075|
|            BURGLARY| 57461|
|ASSAULT WITH DEAD...| 52827|
|THEFT PLAIN - PET...| 52327|
|INTIMATE PARTNER ...| 46310|
|THEFT FROM MOTOR ...| 40804|
+--------------------+------+



In [49]:
# Step 8: Aggregation - Crimes by Location
crimes_by_location = processed_data.groupBy("LOCATION").count().orderBy(desc("count")).limit(10)
print("Top 10 Locations with Most Crimes:")
crimes_by_location.show()




Top 10 Locations with Most Crimes:
+--------------------+-----+
|            LOCATION|count|
+--------------------+-----+
|800 N  ALAMEDA   ...| 2581|
|700 S  FIGUEROA  ...| 1677|
|100    THE GROVE ...| 1635|
|10200    SANTA MO...| 1629|
|6TH              ...| 1569|
|7TH              ...| 1501|
|11800    SANTA MO...| 1486|
|9300    TAMPA    ...| 1445|
|                 7TH| 1427|
|                 6TH| 1357|
+--------------------+-----+



In [50]:
output_path = "/mnt/data/output/crime_analysis_results"
top_categories.write.csv(output_path + "/top_categories", mode="overwrite", header=True)
crimes_by_year.write.csv(output_path + "/crimes_by_year", mode="overwrite", header=True)



# Stop Spark Session
spark.stop()